# 在线教学分析

已知影响学生在线教学适应度（Y: Adaptivity Level）的相关因素包括性别（F1: Gender）、IT专业（F2: IT Student）、经济条件（F3: Financial Condition）、互联网类型（F4: Internet Type）、网络类型（F5: Network Type）与上网设备（F6: Device），利用如表8-4所示数据构建学生在线教学适应度分析模型以对学生在线教学适应度进行预测（Y取值High、Moderate与Low分别表示高、中与低三种类别的适应度），具体要求如下：

1. 构建训练样本（70%）与测试样本（30%）以进行支持向量机模型的训练与测试。

2. 利用交叉验证方式确定支持向量机最优参数并求取最优参数相应的预测精度。

3. 利用主成分析方法对数据进行降维处理并重复步骤2以观察两种情况下模型预测精度的变化。



In [1]:
import pandas as pd  # 导入数据结构与分析库
import numpy as np  # 导入科学计算库
from sklearn.svm import SVC  #导入SVM分类模块
from sklearn.model_selection import train_test_split  # 导入数据划分模型
from sklearn.model_selection import GridSearchCV  # 导入网格式参数调优模块
from sklearn.preprocessing import StandardScaler  # 导入特征标准化库
from sklearn.decomposition import PCA  # 导入主成分分析模块

In [2]:
# 加载数据
data = pd.read_csv('./students_adaptability_level_online_education.csv', encoding='gbk')

In [3]:
# 数据编码
edu_encoding = {
    'Gender': {
        'Boy': 1,
        'Girl': 0
    },
    'Education Level': {
        'University': 2,
        'College': 1,
        'School': 0
    },
    'IT Student': {
        'No': 0,
        'Yes': 1
    },
    'Financial Condition': {
        'Poor': 0,
        'Mid': 1,
        'Rich': 2
    },
    'Network Type': {
        '4G': 2,
        '3G': 1,
        '2G': 0
    },
    'Internet Type': {
        'Wifi': 1,
        'Mobile Data': 0
    },
    'Adaptivity Level': {
        'Low': 0,
        'Moderate': 1,
        'High': 2
    },
    'Device': {
        'Tab': 1,
        'Mobile': 0,
        'Computer': 2
    }
}
for column in data:
    if column in edu_encoding.keys():
        try:
            data[column] = data[column].apply(lambda x: edu_encoding[column][x])
        except:
            print(f"Skipped {column}")

In [4]:
data

,Gender,Education Level,IT Student,Financial Condition,Internet Type,Network Type,Device,Adaptivity Level
0,0,0,0,1,0,2,0,1
1,0,1,0,1,1,2,0,1
2,0,0,0,1,0,1,0,1
3,0,1,0,1,0,2,0,0
4,1,2,1,1,0,2,0,1
5,0,1,0,1,1,2,0,1
6,1,0,0,0,0,1,0,0
7,0,2,1,1,1,1,0,1
8,0,1,1,1,0,2,0,1
9,0,0,0,0,0,1,0,2


In [5]:
# 显示数据基本信息（样本数与特征数）
print('数据基本信息:', data.shape)
x = data.drop('Adaptivity Level', axis=1)
y = data['Adaptivity Level']

数据基本信息: (40, 8)


In [6]:
# 数据标准化
scaler = StandardScaler()
x_ = scaler.fit_transform(x)

In [7]:
# 将数据划分为训练数据与测试数据
x_train, x_test, y_train, y_test = train_test_split(x_, y, test_size=0.3)
print("原始数据类别分布:\n", data['Adaptivity Level'].value_counts())
print("训练集类别分布:\n", y_train.value_counts())

原始数据类别分布:
 Adaptivity Level
1    24
0    15
2     1
Name: count, dtype: int64
训练集类别分布:
 Adaptivity Level
1    18
0    10
Name: count, dtype: int64


In [8]:
# 最优超参数组合列表, C (Regularization Parameter) —— 惩罚系数，越小为 宽容/软间隔 ，越大为 严厉/硬间隔；
param_grid = [
    {'kernel': ['linear'], 'C': [1, 5, 10, 20, 30, 50, 100]},
    {'kernel': ['poly'], 'C': [1], 'degree': [2, 3, 4]},  # 多项式核 degree (多项式次数) —— 仅用于 kernel='poly'
    {'kernel': ['rbf'], 'C': [1, 5, 10, 20, 30, 50, 100], 'gamma': [1, 0.1, 0.01, 0.001]}  # 高斯核； gamma (γ) 是 SVM 中 rbf (高斯核)、poly (多项式核) 和 sigmoid 核函数的一个关键参数。
]

In [9]:
# 构建SVM分类器
model = SVC()
# 通过交叉验证确定最优参数
grid_search = GridSearchCV(model, param_grid, cv=3)
grid_search.fit(x_train, y_train)
# 显示参数优化结果
print('最优模型:', grid_search.best_estimator_)
print('最优参数:', grid_search.best_params_)
print('最高分值:', grid_search.best_score_)

最优模型: SVC(C=1, gamma=0.1)
最优参数: {'C': 1, 'gamma': 0.1, 'kernel': 'rbf'}
最高分值: 0.8222222222222223


In [10]:
# 最优SVM分类器
opt_model = model.set_params(**grid_search.best_params_)
print(grid_search.best_params_)
# 或者opt_model = grid_search.best_estimator_
# 训练SVM分类器
opt_model.fit(x_train,y_train)
# 输出预测精度
print('预测精度:',opt_model.score(x_test,y_test))

{'C': 1, 'gamma': 0.1, 'kernel': 'rbf'}
预测精度: 0.4166666666666667


In [11]:
# 主成分分析 (PCA, Principal Component Analysis) 是一种最常用的数据降维和特征提取技术。
# n_components=0.95 意思是：我要保留原始数据 95% 的信息量
# whiten=True 意思是：白化处理，让每个主成分的方差都变成 1（标准化）
pca = PCA(n_components=0.95, whiten=True).fit(x_train)
x_train_pca = pca.transform(x_train)
x_test_pca = pca.transform(x_test)

In [12]:
# 构建SVM分类器
model = SVC()
# 通过交叉验证确定最优C值
grid_search = GridSearchCV(model,param_grid,cv=5)
grid_search.fit(x_train_pca, y_train)
# 显示参数优化结果
print('最优模型(PCA):',grid_search.best_estimator_)
print('最优参数(PCA):',grid_search.best_params_)
print('最高分值(PCA):',grid_search.best_score_)

最优模型(PCA): SVC(C=1, gamma=0.1)
最优参数(PCA): {'C': 1, 'gamma': 0.1, 'kernel': 'rbf'}
最高分值(PCA): 0.8133333333333335


In [13]:
# 采用最优参数进行模型训练与测试
opt_model = grid_search.best_estimator_
opt_model.fit(x_train_pca,y_train)
# 输出测试精度
print("预测精度(PCA):", opt_model.score(x_test_pca,y_test))

预测精度(PCA): 0.4166666666666667


# 为什么需要 PCA ？

- 消除特征间的相关性（去冗余）
- 降噪与防止过拟合
- 加速训练（针对大数据）